In [1]:
import torch
import pandas as pd
from pathlib import Path

from src.prompt_manager import PromptManager
from src.data_manager import DataManager

from src import data_processing
import yaml
from src import paths
import hashlib

from src.analysis.reliability import ReliabilityAnalyzer

KeyboardInterrupt: 

In [ ]:
with open('../src/configs/config.yaml', 'r') as f:
    full_config = yaml.safe_load(f)

active_analysis = 'mcgill_qa_feedback'
analysis_config = full_config['analyses'][active_analysis]

# active_analysis = full_config['active_analysis']
raw_data_filename = analysis_config['input_filenames']['raw_data_filename']
model_vars = analysis_config['model_vars']
experimental_groups = model_vars['experimental_groups']

In [ ]:
raw_df = pd.read_parquet(paths.RAW_DATA_DIR / f'{raw_data_filename}.parquet')

In [ ]:
final_df, dirty_df = data_processing.get_analysis_ready_df(full_config=full_config,
                                                           active_analysis=active_analysis,
                                                           use_cache=False,
                                                           force_refresh=False,
                                                           return_dirty_df=True,
                                                           balance_experimental_trials=False)

Loading files for analysis mcgill_qa_feedback
🐢 Running full processing pipeline...
Results dir: C:\Users\Wouter Barter\Documents\AI_thesis\results\sandbox\MCGILL_QA_FEEDBACK
input_data_path: C:\Users\Wouter Barter\Documents\AI_thesis\data\processed\MCGILL_QA_FEEDBACK_V1.parquet
⚠️ Skipping archive: No .pt files found in C:\Users\Wouter Barter\Documents\AI_thesis\results\sandbox\MCGILL_QA_FEEDBACK\archive
Finished loading experiment data
Found 0 experimental trials contaminated by garbage output.
Found 1 different scale sizes: [np.int64(4)]
  Processed 67944 rows with scale_size=4
Computing entropy for 1 different scale sizes: [np.int64(4)]
  Processed 67944 rows with scale_size=4


In [ ]:
analyzer = ReliabilityAnalyzer(final_df, group_cols=[
                               'model_name', 'prompt_id', 'dimension_name'], llm_rating_col='mean_rating')
analyzer.compute_reliability_gap(metric='spearman')

,model_name,prompt_id,dimension_name,metric_type,spearman_gap,spearman_hh,spearman_llm_avg
7,Qwen/Qwen3.5-4B,f6db92fc03f8,Relevance,Spearman,-0.019181,0.560768,0.579950
5,Qwen/Qwen3.5-4B,f6db92fc03f8,Completeness,Spearman,-0.007143,0.560768,0.567911
4,Qwen/Qwen3.5-4B,37cf3320a0,quality,Spearman,-0.005112,0.560768,0.565881
3,Qwen/Qwen3-4B-Instruct-2507,f6db92fc03f8,Relevance,Spearman,0.001735,0.560768,0.559034
6,Qwen/Qwen3.5-4B,f6db92fc03f8,Directness,Spearman,0.013932,0.560768,0.546837
2,Qwen/Qwen3-4B-Instruct-2507,f6db92fc03f8,Directness,Spearman,0.017705,0.560768,0.543064
1,Qwen/Qwen3-4B-Instruct-2507,f6db92fc03f8,Completeness,Spearman,0.021082,0.560768,0.539686
0,Qwen/Qwen3-4B-Instruct-2507,37cf3320a0,quality,Spearman,0.042388,0.560768,0.518380
11,meta-llama/Llama-3.2-3B-Instruct,f6db92fc03f8,Relevance,Spearman,0.098223,0.560768,0.462546
10,meta-llama/Llama-3.2-3B-Instruct,f6db92fc03f8,Directness,Spearman,0.130821,0.560768,0.429947


In [ ]:
analyzer = ReliabilityAnalyzer(
    final_df, group_cols=['model_name', 'prompt_id', 'dimension_name'])
analyzer.analyze_calibration('human_disagreement')

,model_name,prompt_id,dimension_name,calibration_corr
0,Qwen/Qwen3-4B-Instruct-2507,37cf3320a0,quality,0.163915
1,Qwen/Qwen3-4B-Instruct-2507,f6db92fc03f8,Completeness,0.190042
2,Qwen/Qwen3-4B-Instruct-2507,f6db92fc03f8,Directness,0.176530
3,Qwen/Qwen3-4B-Instruct-2507,f6db92fc03f8,Relevance,0.203229
4,Qwen/Qwen3.5-4B,37cf3320a0,quality,0.205458
5,Qwen/Qwen3.5-4B,f6db92fc03f8,Completeness,0.184894
6,Qwen/Qwen3.5-4B,f6db92fc03f8,Directness,0.191226
7,Qwen/Qwen3.5-4B,f6db92fc03f8,Relevance,0.194121
8,meta-llama/Llama-3.2-3B-Instruct,37cf3320a0,quality,-0.013996
9,meta-llama/Llama-3.2-3B-Instruct,f6db92fc03f8,Completeness,0.023035


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import scipy.stats as stats

In [ ]:
full_model_df = final_df[['input_id', 'prompt_id', 'model_name', 'mean_human_rating',
                          'mode_rating', 'mean_rating', 'dimension_name', 'normalized_entropy']]

In [ ]:
pm = PromptManager(folder=Path(
    "../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK"))
pm.load_all()
prompt_hash_map = {key: item.metadata['description']
                   for key, item in pm.suites.items()}

PromptManager initialized with folder: ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK
Scanning 2 suites from ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK...
Loaded 2 PromptSuites


In [ ]:
models = {}

for group_key, group_df in full_model_df.groupby(['model_name', 'prompt_id']):
    if group_key[1] not in prompt_hash_map.keys():
        print("Key not in prompts, skipping.")
        continue
    if group_df['dimension_name'].unique()[0] == 'quality':
        formula = 'mean_human_rating ~ mean_rating'
        group_df_wide = group_df
    else:
        group_df_wide = group_df.pivot_table(index=['input_id', 'mean_human_rating', 'model_name'],
                                             columns='dimension_name',
                                             values=['normalized_entropy', 'mean_rating']).reset_index()
        group_df_wide.columns = ['_'.join(col).strip(
            '_') if col[1] else col[0] for col in group_df_wide.columns.values]
        formula = "mean_human_rating ~ mean_rating_Relevance + mean_rating_Completeness + mean_rating_Directness"

    model = smf.ols(formula, data=group_df_wide).fit()

    models[str(f"{group_key[0]}_{prompt_hash_map[group_key[1]]}")] = model

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output


def interactive_dataframe_selector(data_dict, description="Select option:"):
    # label -> value pairs; value is the tuple key
    options = [k for k in data_dict.keys()]
    dropdown = widgets.Dropdown(
        options=options,
        description=description,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    def update_table(change):
        clear_output(wait=True)
        display(dropdown)
        display(data_dict[change.new].summary())

    dropdown.observe(update_table, names='value')
    display(dropdown)
    display(data_dict[dropdown.value].summary())

In [ ]:
models

{'Qwen/Qwen3-4B-Instruct-2507_naive holistic': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1aa60bb9590>,
 'Qwen/Qwen3-4B-Instruct-2507_naive formative': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1aa60d3cc90>,
 'Qwen/Qwen3.5-4B_naive holistic': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1aa6b9c2010>,
 'Qwen/Qwen3.5-4B_naive formative': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1aa60d3f210>,
 'meta-llama/Llama-3.2-3B-Instruct_naive holistic': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1aa60b7c350>,
 'meta-llama/Llama-3.2-3B-Instruct_naive formative': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x1aa60ee3fd0>}

In [ ]:
interactive_dataframe_selector(models)

Dropdown(description='Select option:', index=3, layout=Layout(width='400px'), options=('Qwen/Qwen3-4B-Instruct…

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      mean_human_rating   R-squared:                       0.430
Model:                            OLS   Adj. R-squared:                  0.430
Method:                 Least Squares   F-statistic:                     1422.
Date:                Thu, 19 Mar 2026   Prob (F-statistic):               0.00
Time:                        17:00:13   Log-Likelihood:                -6953.2
No. Observations:                5659   AIC:                         1.391e+04
Df Residuals:                    5655   BIC:                         1.394e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                    0.3354      0.037      9.121      0.000       0.263       0.408
mean_rating_Relevance        0.3766      0.039      9.674      0.000       0.300       0.453
mean_rating_Completeness     0.3395      0.040      8.456      0.000       0.261       0.418
mean_rating_Directness       0.0712      0.033      2.188      0.029       0.007       0.135
==============================================================================
Omnibus:                       13.778   Durbin-Watson:                   1.987
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               13.737
Skew:                          -0.111   Prob(JB):                      0.00104
Kurtosis:                       2.905   Cond. No.                         24.6
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd

models = {}

for group_key, group_df in full_model_df.groupby(['model_name', 'prompt_id']):
    model_name = group_key[0]
    prompt_hash = group_key[1]

    if prompt_hash not in prompt_hash_map:
        print(f"Key {prompt_hash} not in prompts, skipping.")
        continue

    # E.g., 'naive holistic' or 'naive formative'
    prompt_type = prompt_hash_map[prompt_hash]

    # Initialize the nested dictionary for this model if it doesn't exist
    if model_name not in models:
        models[model_name] = {}

    if group_df['dimension_name'].unique()[0] == 'quality':
        formula = 'mean_human_rating ~ mean_rating'
        group_df_wide = group_df
    else:
        group_df_wide = group_df.pivot_table(index=['input_id', 'mean_human_rating', 'model_name'],
                                             columns='dimension_name',
                                             values=['normalized_entropy', 'mean_rating']).reset_index()
        group_df_wide.columns = ['_'.join(col).strip(
            '_') if col[1] else col[0] for col in group_df_wide.columns.values]
        formula = "mean_human_rating ~ mean_rating_Relevance + mean_rating_Completeness + mean_rating_Directness"

    model = smf.ols(formula, data=group_df_wide).fit()

    # Store it under the specific model and prompt type
    models[model_name][prompt_type] = model

In [ ]:
from IPython.display import display


def compare_formative_vs_holistic(nested_models_dict, holistic_key='naive holistic', formative_key='naive formative'):
    rows = []

    for model_name, runs in nested_models_dict.items():
        # Check if we have both runs to make a comparison
        if holistic_key in runs and formative_key in runs:
            mod_h = runs[holistic_key]
            mod_f = runs[formative_key]

            # Extract R2
            r2_h = mod_h.rsquared
            r2_f = mod_f.rsquared

            # Extract Adjusted R2
            adj_r2_h = mod_h.rsquared_adj
            adj_r2_f = mod_f.rsquared_adj

            # Calculate Absolute Differences
            diff_r2 = r2_f - r2_h
            diff_adj_r2 = adj_r2_f - adj_r2_h

            # Calculate Relative Improvements (%)
            # Formula: (New - Old) / abs(Old)
            rel_r2 = (diff_r2 / abs(r2_h)) if r2_h != 0 else 0
            rel_adj_r2 = (diff_adj_r2 / abs(adj_r2_h)) if adj_r2_h != 0 else 0

            rows.append({
                'Model': model_name,
                'Holistic R²': r2_h,
                'Formative R²': r2_f,
                'Δ R² (Abs)': diff_r2,
                'Δ R² (Rel %)': rel_r2 * 100,
                'Holistic Adj R²': adj_r2_h,
                'Formative Adj R²': adj_r2_f,
                'Δ Adj R² (Abs)': diff_adj_r2,
                'Δ Adj R² (Rel %)': rel_adj_r2 * 100
            })
        else:
            print(
                f"⚠️ Skipping {model_name}: Missing either '{holistic_key}' or '{formative_key}'")

    # Compile and format
    df_compare = pd.DataFrame(rows)

    # Sort by the best relative Adjusted R2 improvement
    if not df_compare.empty:
        df_compare = df_compare.sort_values(
            'Δ Adj R² (Rel %)', ascending=False).reset_index(drop=True)

    return df_compare


# --- EXECUTION ---
comparison_df = compare_formative_vs_holistic(models)

# Styler to make the improvements pop


def style_improvements(val):
    if pd.isna(val):
        return ''
    if val > 0:
        return 'color: green; font-weight: bold'
    if val < 0:
        return 'color: red'
    return ''


styled_comparison = comparison_df.style\
    .applymap(style_improvements, subset=['Δ R² (Abs)', 'Δ R² (Rel %)', 'Δ Adj R² (Abs)', 'Δ Adj R² (Rel %)'])\
    .format({
        'Holistic R²': '{:.4f}',
        'Formative R²': '{:.4f}',
        'Δ R² (Abs)': '{:.4f}',
        'Δ R² (Rel %)': '{:.2f}%',
        'Holistic Adj R²': '{:.4f}',
        'Formative Adj R²': '{:.4f}',
        'Δ Adj R² (Abs)': '{:.4f}',
        'Δ Adj R² (Rel %)': '{:.2f}%'
    })

display(styled_comparison)

C:\Users\Wouter Barter\AppData\Local\Temp\ipykernel_19784\3909031213.py:61: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(style_improvements, subset=['Δ R² (Abs)', 'Δ R² (Rel %)', 'Δ Adj R² (Abs)', 'Δ Adj R² (Rel %)'])\


,Model,Holistic R²,Formative R²,Δ R² (Abs),Δ R² (Rel %),Holistic Adj R²,Formative Adj R²,Δ Adj R² (Abs),Δ Adj R² (Rel %)
0,meta-llama/Llama-3.2-3B-Instruct,0.1902,0.2745,0.0843,44.34%,0.1901,0.2741,0.0841,44.25%
1,Qwen/Qwen3-4B-Instruct-2507,0.2768,0.3647,0.0878,31.73%,0.2767,0.3644,0.0876,31.67%
2,Qwen/Qwen3.5-4B,0.4102,0.4301,0.0199,4.84%,0.4101,0.4298,0.0197,4.79%


------------

# Combining formative scores into latent scores

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd

# 1. Expand the slice to include the human baselines and disagreement metrics
cols_to_keep = [
    'input_id', 'prompt_id', 'model_name', 'mean_human_rating',
    'mode_rating', 'mean_rating', 'dimension_name', 'normalized_entropy',
    'score_1', 'score_2', 'human_disagreement', 'entropy'  # <--- Added these back!
]
# Only grab columns that actually exist to avoid KeyErrors
existing_cols = [c for c in cols_to_keep if c in final_df.columns]
full_model_df = final_df[existing_cols].copy()

models = {}
processed_dfs = []

# 2. Define the columns that represent a single unique deal/trial
# These go into the index so they survive the pivot!
index_cols = ['input_id', 'prompt_id', 'model_name', 'mean_human_rating']
for extra in ['score_1', 'score_2', 'human_disagreement']:
    if extra in full_model_df.columns:
        index_cols.append(extra)

# 3. The Generation Loop
for group_key, group_df in full_model_df.groupby(['model_name', 'prompt_id']):
    model_name = group_key[0]
    prompt_hash = group_key[1]

    if prompt_hash not in prompt_hash_map.keys():
        continue

    prompt_type = prompt_hash_map[prompt_hash]

    if group_df['dimension_name'].unique()[0] == 'quality':
        # --- HOLISTIC ---
        formula = 'mean_human_rating ~ mean_rating'
        group_df_wide = group_df.copy()
    else:
        # --- FORMATIVE ---
        group_df_wide = group_df.pivot_table(
            index=index_cols,
            columns='dimension_name',
            # Pivot entropy too if it exists
            values=['normalized_entropy', 'mean_rating', 'entropy']
        ).reset_index()

        # Flatten the multi-level columns from the pivot
        group_df_wide.columns = ['_'.join(col).strip(
            '_') if col[1] else col[0] for col in group_df_wide.columns.values]
        formula = "mean_human_rating ~ mean_rating_Relevance + mean_rating_Completeness + mean_rating_Directness"

        # To analyze calibration for Formative, we need a single 'entropy' score.
        # We average the entropy of the sub-dimensions to represent total uncertainty.
        entropy_cols = [
            c for c in group_df_wide.columns if c.startswith('entropy_')]
        if entropy_cols:
            group_df_wide['entropy'] = group_df_wide[entropy_cols].mean(axis=1)

    # Fit the model and generate the latent score
    model = smf.ols(formula, data=group_df_wide).fit()
    models[f"{model_name}_{prompt_type}"] = model

    # Generate the prediction (the optimal weighted score)
    group_df_wide['latent_score'] = model.predict(group_df_wide)
    group_df_wide['prompt_type'] = prompt_type

    processed_dfs.append(group_df_wide)

# The master dataframe now has holistic and formative scores perfectly aligned!
master_latent_df = pd.concat(processed_dfs, ignore_index=True)

In [ ]:
master_latent_df

,input_id,prompt_id,model_name,mean_human_rating,mode_rating,mean_rating,dimension_name,normalized_entropy,score_1,score_2,...,prompt_type,entropy_Completeness,entropy_Directness,entropy_Relevance,mean_rating_Completeness,mean_rating_Directness,mean_rating_Relevance,normalized_entropy_Completeness,normalized_entropy_Directness,normalized_entropy_Relevance
0,eff9e000675931f5,37cf3320a0,Qwen/Qwen3-4B-Instruct-2507,3.0,1.0,1.000000,quality,8.061814e-08,4,2,...,naive holistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,c8be913323f10444,37cf3320a0,Qwen/Qwen3-4B-Instruct-2507,4.0,1.0,1.001135,quality,6.372732e-03,4,4,...,naive holistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,31effc925bc04105,37cf3320a0,Qwen/Qwen3-4B-Instruct-2507,2.0,1.0,1.000000,quality,2.980207e-06,1,3,...,naive holistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,610d0764ed04054d,37cf3320a0,Qwen/Qwen3-4B-Instruct-2507,2.5,4.0,4.000000,quality,8.243048e-06,2,3,...,naive holistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,033e9fcef5d75297,37cf3320a0,Qwen/Qwen3-4B-Instruct-2507,1.5,1.0,1.000373,quality,2.357447e-03,1,2,...,naive holistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33949,ffbb07bd15a997f7,f6db92fc03f8,meta-llama/Llama-3.2-3B-Instruct,2.0,NaN,NaN,NaN,NaN,3,1,...,naive formative,0.945916,1.039791,1.203454,2.866829,2.246794,2.871162,0.682334,0.750051,0.868109
33950,ffbee8c0a0f5c10f,f6db92fc03f8,meta-llama/Llama-3.2-3B-Instruct,2.5,NaN,NaN,NaN,NaN,2,3,...,naive formative,0.699828,0.971037,0.803559,3.291280,2.916329,3.534163,0.504819,0.700455,0.579645
33951,ffc417dece35455f,f6db92fc03f8,meta-llama/Llama-3.2-3B-Instruct,1.5,NaN,NaN,NaN,NaN,2,1,...,naive formative,0.801348,1.081394,0.973851,3.399640,2.798410,3.422068,0.578051,0.780061,0.702485
33952,ffc9922f0a9616b5,f6db92fc03f8,meta-llama/Llama-3.2-3B-Instruct,4.0,NaN,NaN,NaN,NaN,4,4,...,naive formative,0.736016,1.213606,0.850306,3.654136,2.795226,3.581684,0.530923,0.875432,0.613366


In [ ]:
# Initialize the analyzer using the combined Latent Score
latent_analyzer = ReliabilityAnalyzer(
    df=master_latent_df,
    group_cols=['model_name', 'prompt_type'],
    llm_rating_col='latent_score',
    col_map={
        'h1': 'score_1',
        'h2': 'score_2',
        'llm': 'latent_score'
    }
)

# 1. Compute Reliability Gap (Spearman)
print("--- RELIABILITY GAP (SPEARMAN) ---")
reliability_results = latent_analyzer.compute_reliability_gap(
    metric='spearman')
display(reliability_results)

# 2. Analyze Calibration (Uncertainty vs. Difficulty)
print("\n--- CALIBRATION (ENTROPY VS. DISAGREEMENT) ---")
calibration_results = latent_analyzer.analyze_calibration(
    disagreement_col='human_disagreement')
display(calibration_results)

--- RELIABILITY GAP (SPEARMAN) ---


,model_name,prompt_type,metric_type,spearman_gap,spearman_hh,spearman_llm_avg
2,Qwen/Qwen3.5-4B,naive formative,Spearman,-0.020744,0.560768,0.581512
3,Qwen/Qwen3.5-4B,naive holistic,Spearman,-0.005112,0.560768,0.565881
0,Qwen/Qwen3-4B-Instruct-2507,naive formative,Spearman,-0.000099,0.560768,0.560868
1,Qwen/Qwen3-4B-Instruct-2507,naive holistic,Spearman,0.042388,0.560768,0.518380
4,meta-llama/Llama-3.2-3B-Instruct,naive formative,Spearman,0.097873,0.560768,0.462896
5,meta-llama/Llama-3.2-3B-Instruct,naive holistic,Spearman,0.179496,0.560768,0.381272



--- CALIBRATION (ENTROPY VS. DISAGREEMENT) ---


,model_name,prompt_type,calibration_corr
0,Qwen/Qwen3-4B-Instruct-2507,naive formative,0.197388
1,Qwen/Qwen3-4B-Instruct-2507,naive holistic,0.163915
2,Qwen/Qwen3.5-4B,naive formative,0.199626
3,Qwen/Qwen3.5-4B,naive holistic,0.205458
4,meta-llama/Llama-3.2-3B-Instruct,naive formative,0.053060
5,meta-llama/Llama-3.2-3B-Instruct,naive holistic,-0.013996
